In [21]:
from dotenv import load_dotenv
import os

load_dotenv()

print("URI:", os.getenv("NEO4J_URI"))

URI: neo4j+s://579b613f.databases.neo4j.io


In [22]:
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
GROQ_API_KEY = os.environ["GROQ_API_KEY"]

1. Connect to Neo4j

In [23]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

print("Connected to Neo4j")

Connected to Neo4j


2. Create the LLM

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    reasoning_effort="low",
    model_kwargs={
        "parallel_tool_calls": False
    }
)

3. Clear the old graph

In [25]:
graph.query("MATCH (n) DETACH DELETE n")
print("Database cleared")

Database cleared


4. Dataset created

In [26]:
from langchain_core.documents import Document

movie_texts = [
    "J.R.R. Tolkien wrote The Hobbit. The Hobbit is a Fantasy book published in 1937. Peter Jackson directed the film adaptation of The Hobbit. Martin Freeman acted in The Hobbit.",
    "J.R.R. Tolkien wrote The Lord of the Rings. The Lord of the Rings is a Fantasy book published in 1954. Peter Jackson directed the film adaptation of The Lord of the Rings. Elijah Wood acted in The Lord of the Rings.",
    "Peter Jackson directed King Kong. King Kong is an Adventure film released in 2005. Andy Serkis acted in King Kong. Andy Serkis also acted in The Lord of the Rings."
]

documents = [Document(page_content=text) for text in movie_texts]

chunks = documents

for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c.page_content}")

Chunk 1: J.R.R. Tolkien wrote The Hobbit. The Hobbit is a Fantasy book published in 1937. Peter Jackson directed the film adaptation of The Hobbit. Martin Freeman acted in The Hobbit.
Chunk 2: J.R.R. Tolkien wrote The Lord of the Rings. The Lord of the Rings is a Fantasy book published in 1954. Peter Jackson directed the film adaptation of The Lord of the Rings. Elijah Wood acted in The Lord of the Rings.
Chunk 3: Peter Jackson directed King Kong. King Kong is an Adventure film released in 2005. Andy Serkis acted in King Kong. Andy Serkis also acted in The Lord of the Rings.


5. Extract graph data

In [27]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Book", "Movie", "Genre"],
    allowed_relationships=[
        ("Person", "WROTE", "Book"),
        ("Book", "ADAPTED_INTO", "Movie"),
        ("Person", "DIRECTED", "Movie"),
        ("Person", "ACTED_IN", "Movie"),
        ("Book", "IN_GENRE", "Genre"),
        ("Movie", "IN_GENRE", "Genre"),
    ],
)



In [28]:
graph_documents = llm_transformer.convert_to_graph_documents(chunks)

In [29]:
for i, gd in enumerate(graph_documents):
    print(f"\n--- Graph Document {i+1} ---")
    for n in gd.nodes:
        print("NODE:", n.id, f"({n.type})")
    for r in gd.relationships:
        print("REL: ", r.source.id, f"-[{r.type}]->", r.target.id)


--- Graph Document 1 ---
NODE: J.R.R. Tolkien (Person)
NODE: The Hobbit (Book)
NODE: Peter Jackson (Person)
NODE: Martin Freeman (Person)
NODE: The Hobbit (Film) (Movie)
NODE: Fantasy (Genre)
REL:  J.R.R. Tolkien -[WROTE]-> The Hobbit
REL:  The Hobbit -[ADAPTED_INTO]-> The Hobbit (Film)
REL:  Peter Jackson -[DIRECTED]-> The Hobbit (Film)
REL:  Martin Freeman -[ACTED_IN]-> The Hobbit (Film)
REL:  The Hobbit -[IN_GENRE]-> Fantasy

--- Graph Document 2 ---
NODE: J.R.R. Tolkien (Person)
NODE: The Lord Of The Rings (Book)
NODE: Fantasy (Genre)
NODE: Peter Jackson (Person)
NODE: The Lord Of The Rings (Film) (Movie)
NODE: Elijah Wood (Person)
REL:  J.R.R. Tolkien -[WROTE]-> The Lord Of The Rings
REL:  The Lord Of The Rings -[IN_GENRE]-> Fantasy
REL:  The Lord Of The Rings -[ADAPTED_INTO]-> The Lord Of The Rings (Film)
REL:  Peter Jackson -[DIRECTED]-> The Lord Of The Rings (Film)
REL:  Elijah Wood -[ACTED_IN]-> The Lord Of The Rings (Film)

--- Graph Document 3 ---
NODE: Peter Jackson (Perso

6. Store in Neo4j

In [30]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True,
)

graph.refresh_schema()
print(graph.schema)

Node properties:
Person {id: STRING}
Movie {id: STRING}
Document {id: STRING, text: STRING}
Genre {id: STRING}
Book {id: STRING}
Relationship properties:

The relationships:
(:Person)-[:WROTE]->(:Book)
(:Person)-[:WROTE]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:ACTED_IN]->(:Book)
(:Movie)-[:IN_GENRE]->(:Genre)
(:Movie)-[:ADAPTED_INTO]->(:Movie)
(:Document)-[:MENTIONS]->(:Person)
(:Document)-[:MENTIONS]->(:Book)
(:Document)-[:MENTIONS]->(:Movie)
(:Document)-[:MENTIONS]->(:Genre)
(:Book)-[:IN_GENRE]->(:Genre)
(:Book)-[:ADAPTED_INTO]->(:Movie)


7. Custom prompts

In [31]:
from langchain_core.prompts import PromptTemplate

CYPHER_TEMPLATE = """Task: Generate a Cypher query for a Neo4j graph database.

Rules:
- Every node has only an `id` property (e.g. (:Person {{id: "Peter Jackson"}})).
- Never use `name`, `title`, or `released`. Only use `id`.
- Titles may carry a suffix like " (Film)", " (Movie)" or " (Book)". Match titles with CONTAINS, not equality.
- When returning a Book or Movie id, clean it with:
  trim(replace(replace(replace(x.id, " (Film)", ""), " (Movie)", ""), " (Book)", ""))
- Match ids case-insensitively using toLower().
- Use only the node labels and relationship types in the schema.
- Return only the Cypher query, with no explanation and no markdown.

Schema:
{schema}

Examples:
Question: Which movies did Peter Jackson direct?
Cypher: MATCH (p:Person)-[:DIRECTED]->(m:Movie) WHERE toLower(p.id) = toLower("Peter Jackson") RETURN DISTINCT trim(replace(replace(replace(m.id, " (Film)", ""), " (Movie)", ""), " (Book)", "")) AS movie

Question: Which books did J.R.R. Tolkien write?
Cypher: MATCH (p:Person)-[:WROTE]->(b:Book) WHERE toLower(p.id) = toLower("J.R.R. Tolkien") RETURN DISTINCT trim(replace(replace(replace(b.id, " (Film)", ""), " (Movie)", ""), " (Book)", "")) AS book

Question: Which actor appeared in both King Kong and The Lord of the Rings?
Cypher: MATCH (p:Person)-[:ACTED_IN]->(m1:Movie), (p)-[:ACTED_IN]->(m2:Movie) WHERE toLower(m1.id) CONTAINS toLower("King Kong") AND toLower(m2.id) CONTAINS toLower("The Lord of the Rings") RETURN DISTINCT p.id AS actor

Question: Who directed the film adaptations of books written by Tolkien?
Cypher: MATCH (a:Person)-[:WROTE]->(b:Book)-[:ADAPTED_INTO]->(m:Movie)<-[:DIRECTED]-(d:Person) WHERE toLower(a.id) CONTAINS toLower("Tolkien") RETURN DISTINCT d.id AS director

Question: {question}
Cypher:"""

CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template=CYPHER_TEMPLATE,
)

QA_TEMPLATE = """Answer the question using ONLY the context below.
If the context is empty or does not contain the answer, reply exactly: "I don't know based on the graph."
Do not use outside knowledge.

Context:
{context}

Question:
{question}
"""

QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=QA_TEMPLATE,
)

8. Build the chain

In [32]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=CYPHER_PROMPT,
    qa_prompt=QA_PROMPT,
    verbose=True,
    allow_dangerous_requests=True,
)

In [33]:
questions = [
    "Which films did Peter Jackson direct?",
    "Which actor appeared in both King Kong and The Lord of the Rings?",
    "Which books did J.R.R. Tolkien write?",
    "Who directed the film adaptations of books written by Tolkien?",
    "Who directed Titanic?",   # not in the graph, so it should say "I don't know"
]

for q in questions:
    response = chain.invoke({"query": q})
    print(f"Q: {q}")
    print(f"A: {response['result']}\n")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie) WHERE toLower(p.id) = toLower("Peter Jackson") RETURN DISTINCT trim(replace(replace(replace(m.id, " (Film)", ""), " (Movie)", ""), " (Book)", "")) AS movie
Full Context:
[{'movie': 'The Hobbit'}, {'movie': 'The Lord Of The Rings'}, {'movie': 'King Kong'}]

> Finished chain.
Q: Which films did Peter Jackson direct?
A: I don't know based on the graph.



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:ACTED_IN]->(m1:Movie), (p)-[:ACTED_IN]->(m2:Movie) WHERE toLower(m1.id) CONTAINS toLower("King Kong") AND toLower(m2.id) CONTAINS toLower("The Lord of the Rings") RETURN DISTINCT p.id AS actor
Full Context:
[{'actor': 'Andy Serkis'}]

> Finished chain.
Q: Which actor appeared in both King Kong and The Lord of the Rings?
A: Andy Serkis



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:WROTE]->(b:Book) WHERE toLower(p.id) CONTA